In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
for  dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
from pandas.tseries.offsets import DateOffset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

In [11]:
df = pd.read_csv("/content/Employee 1000x.csv")
df.head()

,Index,First Name,Last Name,Sex,Email,Phone,Date of birth,Job Title
0,1,Sara,Mcguire,Female,tsharp@example.net,(971)643-6089x9160,17-08-21,"Editor, commissioning"
1,2,Alisha,Hebert,Male,vincentgarrett@example.net,+1-114-355-1841x78347,28-06-69,Broadcast engineer
2,3,Gwendolyn,Sheppard,Male,mercadojonathan@example.com,9017807728,25-09-15,Industrial buyer
3,4,Kristine,Mccann,Female,lindsay55@example.com,+1-607-333-9911x59088,27-07-78,Multimedia specialist
4,5,Bobby,Pittman,Female,blevinsmorgan@example.com,3739847538,17-11-89,Planning and development surveyor


In [12]:
print("Data info")
df.info()

Data info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Index          10000 non-null  int64 
 1   First Name     10000 non-null  object
 2   Last Name      10000 non-null  object
 3   Sex            10000 non-null  object
 4   Email          10000 non-null  object
 5   Phone          10000 non-null  object
 6   Date of birth  10000 non-null  object
 7   Job Title      10000 non-null  object
dtypes: int64(1), object(7)
memory usage: 625.1+ KB


In [17]:
#df = df.drop(columns = ["Index", "First Name", "Last Name", "Email", "Phone"])
print(df)

         Sex Date of birth                          Job Title
0     Female      17-08-21              Editor, commissioning
1       Male      28-06-69                 Broadcast engineer
2       Male      25-09-15                   Industrial buyer
3     Female      27-07-78              Multimedia specialist
4     Female      17-11-89  Planning and development surveyor
...      ...           ...                                ...
9995    Male      28-07-74           Scientist, physiological
9996  Female      20-08-32                  Warehouse manager
9997  Female      06-06-66                             Lawyer
9998    Male      09-05-07              Accounting technician
9999  Female      14-08-35                  Drilling engineer

[10000 rows x 3 columns]


In [18]:
df["Date of birth"] = pd.to_datetime(df["Date of birth"], format = "%d-%m-%y",errors = "coerce")

In [20]:
df["Year"] = df["Date of birth"].dt.year
df.head()

,Sex,Date of birth,Job Title,Year
0,Female,2021-08-17,"Editor, commissioning",2021
1,Male,1969-06-28,Broadcast engineer,1969
2,Male,2015-09-25,Industrial buyer,2015
3,Female,1978-07-27,Multimedia specialist,1978
4,Female,1989-11-17,Planning and development surveyor,1989


In [25]:
df["Age"] = 2025 - df["Year"]

df["Original date"] = df["Date of birth"]

df.loc[df["Age"] < 0, "Date of birth"] -= DateOffset(years = 100)


df["Update Age"] = 2025 - df["Date of birth"].dt.year

df.loc[df["Age"]< 0]

,Sex,Date of birth,Job Title,Year,Age,Original date,Update Age
14,Male,1859-02-07,Paediatric nurse,2059,-34,1959-02-07,166
19,Female,1834-01-06,"Engineer, aeronautical",2034,-9,1934-01-06,191
20,Male,1844-09-16,"Research officer, government",2044,-19,1944-09-16,181
21,Male,1844-04-09,Retail buyer,2044,-19,1944-04-09,181
24,Male,1827-08-24,Occupational therapist,2027,-2,1927-08-24,198
...,...,...,...,...,...,...,...
9992,Female,1829-10-18,Restaurant manager,2029,-4,1929-10-18,196
9993,Female,1838-06-15,Control and instrumentation engineer,2038,-13,1938-06-15,187
9996,Female,1832-08-20,Warehouse manager,2032,-7,1932-08-20,193
9997,Female,1866-06-06,Lawyer,2066,-41,1966-06-06,159


In [26]:
df = df.drop(columns = ["Year", "Age", "Original date"])
df.rename(columns = {"Update Age" : "Age"}, inplace=True)

In [28]:
bins = [17,25,35,45,55,100]
labels = ["A", "B", "C", "D", "E"]

df["Age Group"] = pd.cut(df["Age"], bins = bins, labels = labels)

df.head()

,Sex,Date of birth,Job Title,Age,Age Group
0,Female,2021-08-17,"Editor, commissioning",4,NaN
1,Male,1969-06-28,Broadcast engineer,56,E
2,Male,2015-09-25,Industrial buyer,10,NaN
3,Female,1978-07-27,Multimedia specialist,47,D
4,Female,1989-11-17,Planning and development surveyor,36,C


In [29]:

df["Job Category"] = df["Job Title"]


education = df["Job Category"].str.contains(r'\b(professor|teacher)\b', regex = True, case = False)

administration = df["Job Category"].str.contains(r'\b(officer|secretary)\b|\badmin\w*', regex = True, case = False)

healthcare = df["Job Category"].str.contains(r'\b(nurse|health|doctor)\b|psych\w*', regex = True, case = False)

manager = df["Job Category"].str.contains("manager", case = False)

tech = df["Job Category"].str.contains(r'\b(analyst|programmer|scientist)', regex = True, case = False)

engineer = df["Job Category"].str.contains("engineer", case = False)

design = df["Job Category"].str.contains(r'\b(designer|editor)\b', regex = True, case = False)

other = other = df["Job Category"].str.contains(r'\w+', regex = True)



df["Job Category"] = df["Job Category"].case_when([(education, "education"),
                                                   (administration, "administration"),
                                                   (healthcare, "healthcare"),
                                                   (manager, "manager"),
                                                   (tech, "tech"),
                                                   (engineer, "engineer"),
                                                   (design, "design"),
                                                   (other, "other")])

<ipython-input-29-e007a842e496>:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  education = df["Job Category"].str.contains(r'\b(professor|teacher)\b', regex = True, case = False)
<ipython-input-29-e007a842e496>:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  administration = df["Job Category"].str.contains(r'\b(officer|secretary)\b|\badmin\w*', regex = True, case = False)
<ipython-input-29-e007a842e496>:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  healthcare = df["Job Category"].str.contains(r'\b(nurse|health|doctor)\b|psych\w*', regex = True, case = False)
<ipython-input-29-e007a842e496>:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.ex

In [30]:
df.head()

,Sex,Date of birth,Job Title,Age,Age Group,Job Category
0,Female,2021-08-17,"Editor, commissioning",4,NaN,design
1,Male,1969-06-28,Broadcast engineer,56,E,engineer
2,Male,2015-09-25,Industrial buyer,10,NaN,other
3,Female,1978-07-27,Multimedia specialist,47,D,other
4,Female,1989-11-17,Planning and development surveyor,36,C,other


In [31]:

adult = df[df["Age"] >= 18]

mean_age_category = adult.groupby("Job Category")["Age"].mean()


def solve_age (row):
    if row["Age"] <= 18:
        return mean_age_category.get(row["Job Category"], np.nan)
    return row["Age"]


df["Right Age"] = df.apply(solve_age, axis = 1)


df["Age Group"] = pd.cut(df["Right Age"], bins = bins, labels = labels)
df.head()

,Sex,Date of birth,Job Title,Age,Age Group,Job Category,Right Age
0,Female,2021-08-17,"Editor, commissioning",4,NaN,design,114.614987
1,Male,1969-06-28,Broadcast engineer,56,E,engineer,56.000000
2,Male,2015-09-25,Industrial buyer,10,NaN,other,108.085931
3,Female,1978-07-27,Multimedia specialist,47,D,other,47.000000
4,Female,1989-11-17,Planning and development surveyor,36,C,other,36.000000


In [37]:
le = LabelEncoder()
le.fit(df["Sex"])
df["Sex bin"] = le.transform(df["Sex"])

df.head()

,Sex,Date of birth,Job Title,Age,Age Group,Job Category,Right Age,Sex bin,Age Group bin
0,Female,2021-08-17,"Editor, commissioning",4,NaN,design,114.614987,0,5
1,Male,1969-06-28,Broadcast engineer,56,E,engineer,56.000000,1,4
2,Male,2015-09-25,Industrial buyer,10,NaN,other,108.085931,1,5
3,Female,1978-07-27,Multimedia specialist,47,D,other,47.000000,0,3
4,Female,1989-11-17,Planning and development surveyor,36,C,other,36.000000,0,2


In [38]:
le.fit(df["Age Group"])
df["Age Group bin"] = le.transform(df["Age Group"])

df.head()

,Sex,Date of birth,Job Title,Age,Age Group,Job Category,Right Age,Sex bin,Age Group bin
0,Female,2021-08-17,"Editor, commissioning",4,NaN,design,114.614987,0,5
1,Male,1969-06-28,Broadcast engineer,56,E,engineer,56.000000,1,4
2,Male,2015-09-25,Industrial buyer,10,NaN,other,108.085931,1,5
3,Female,1978-07-27,Multimedia specialist,47,D,other,47.000000,0,3
4,Female,1989-11-17,Planning and development surveyor,36,C,other,36.000000,0,2


In [45]:
X = df[["Sex bin", "Age Group bin"]]
y = df["Job Category"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state = 12)

In [46]:
rfc = RandomForestClassifier(random_state = 0, class_weight= "balanced")
modelo = rfc.fit(X_train, y_train)
results = modelo.predict(X_test)

In [47]:
labels = df["Job Category"].unique()
report = classification_report(y_test, results, labels = labels)
print(report)

                precision    recall  f1-score   support

        design       0.05      0.33      0.08       107
      engineer       0.16      0.11      0.13       196
         other       0.00      0.00      0.00      1018
administration       0.00      0.00      0.00       245
     education       0.03      0.43      0.05        51
          tech       0.09      0.05      0.06       125
    healthcare       0.06      0.04      0.05       116
       manager       0.07      0.08      0.08       142

      accuracy                           0.05      2000
     macro avg       0.06      0.13      0.06      2000
  weighted avg       0.03      0.05      0.03      2000



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
